In [ ]:
import scanpy as sc
import pandas as pd
import memento
import numpy as np
import os

In [ ]:
data_path = '/home3/ciervo/scMULTIOME/Analisi/scRNA/memento/'

In [ ]:
# Load matrix
adata = sc.read_mtx(data_path + 'count_mtx.mtx')

# Load metadata
metadata = pd.read_csv(data_path + 'metadata.csv')
adata.obs = metadata

# Load genes
genes = pd.read_csv(data_path + 'genes.csv')
adata.obs_names = adata.obs['Barcode']
adata.var_names = genes['gene']

original_mp = adata.obs['Metaprogram_assignment'].copy()

In [ ]:
for i in range(1, 8):

    sample = f"MP_{i}"
    print(f"\nProcessing {sample} vs rest")

    adata.obs['Metaprogram_assignment'] = original_mp.copy()

    adata.obs['Metaprogram_assignment'] = (
        adata.obs['Metaprogram_assignment'] == sample
    ).astype(int)

    if adata.obs['Metaprogram_assignment'].nunique() < 2:
        print(f"{sample}: only one class present — skipping")
        continue

    result_1d = memento.binary_test_1d(
        adata=adata,
        capture_rate=0.07,
        treatment_col='Metaprogram_assignment',
        num_cpus=5,
        num_boot=5000
    )

    out_file = os.path.join(data_path, f"{sample}.csv")
    result_1d.to_csv(out_file)

    print(f"Saved: {out_file}")

adata.obs['Metaprogram_assignment'] = original_mp